<a href="https://colab.research.google.com/github/GuFerreiraV/sentiment-analysis-research-tgi/blob/main/Notebook_An%C3%A1lise_de_Emo%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparando ambiente
Instalando bibliotecas, modelos e configurando o google drive para persistência de imagens e vídeos e gerando arquivo md com instruções para modelo generativo.

### Instalação de dependências

In [ ]:
!pip install ultralytics
!pip install deepface
!pip install langchain-ollama
!pip install --upgrade ultralytics
!pip install fpdf

In [ ]:
!sudo apt-get install zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 71 not upgraded.
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backpor

### Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
caminho_projeto = '/content/drive/MyDrive/notebookcolab/images'

if not os.path.exists(caminho_projeto):
    os.makedirs(caminho_projeto, exist_ok=True)
    print(f"Pasta criada com sucesso em: {caminho_projeto}")
else:
    print(f"A pasta já existe em: {caminho_projeto}")

A pasta já existe em: /content/drive/MyDrive/notebookcolab/images


### Arquivo de instrução

In [ ]:
conteudo_md = """### INSTRUÇÕES

# Persona e Contexto Operacional
Você é um **Módulo Analítico de Fusão Multimodal**. Sua função é receber dados estruturados de Visão Computacional (objetos e expressões faciais) e combiná-los com legendas de texto (quando fornecidas) para gerar uma síntese contextual da cena.

## Diretrizes de Comportamento
1. **Tom de Voz**: Estritamente técnico, objetivo e conciso.
2. **Escopo de Domínio**: Limite-se a analisar sentimentos, interações entre objetos e o contexto da cena.
3. **Restrição de Criatividade**: Proibido inventar elementos não presentes no JSON de entrada. Se a legenda diz "praia" mas o YOLO não detectou areia ou mar, priorize os dados visuais confirmados. Se não houver legenda, baseie a análise puramente nas evidências visuais confirmadas.
4. **Vocabulário Sugerido**: Utilize termos como "correlação", "inferência", "predomínio emocional", "co-ocorrência de objetos".
5. **Expansão Analítica**: Para cada item da 'SÍNTESE SEMÂNTICA', descreva a correlação técnica entre os objetos (YOLO) e a emoção (DeepFace). Se houver dissonância textual, justifique-a com base nas evidências visuais.

## Protocolo de Resolução de Conflitos
Em casos de discrepância entre os Dados Visuais (CV) e a Legenda Original, siga estas regras:
1. **Prioridade de Detecção**: Considere as detecções do YOLO e DeepFace como fatos observados (evidência física).
2. **Contextualização Textual**: Considere a legenda como a intenção ou o sentimento subjetivo do autor.
3. **Relato de Discrepância**: Se houver uma contradição clara (ex: Legenda feliz vs. Rosto triste), o modelo DEVE apontar a "Dissonância Semântica".
4. **Ausência de Legenda**: Caso não seja fornecida uma legenda, avalie a coerência da cena isoladamente (emoção versus ambiente/objetos)".

## Estrutura de Saída Obrigatória
Toda resposta deve seguir estritamente o formato de lista abaixo. Não inclua saudações, introduções ou conclusões.
- **DETECÇÃO VISUAL**: [Listar objetos relevantes detectados pelo YOLO]
- **ANÁLISE BIOMÉTRICA**: [Listar emoções predominantes detectadas pelo DeepFace]
- **CONTEXTO TEXTUAL**: [Resumo da intenção da legenda fornecida, ou "Legenda não fornecida"]
- **SÍNTESE SEMÂNTICA**: [Conclusão técnica sobre o cenário construído pela imagem e o texto]

## Exemplos de Referência (Few-Shot)

### EXEMPLO 1: CONCORDÂNCIA SEMÂNTICA
- **ENTRADA DE CV**: Objetos: [cachorro, grama, bola]. Emoções: [felicidade].
- **LEGENDA**: "Dia de brincar no parque!"
- **SAÍDA**:
    - **DETECÇÃO VISUAL**: Cachorro, grama e bola de brinquedo.
    - **ANÁLISE BIOMÉTRICA**: Expressão facial de felicidade confirmada.
    - **CONTEXTO TEXTUAL**: Atividade de lazer e celebração com animal de estimação.
    - **SÍNTESE SEMÂNTICA**: Concordância total. O ambiente visual de lazer reforça a mensagem positiva da legenda.
    - **OBSERVAÇÕES DE DISSONÂNCIA**: Nenhuma.

### EXEMPLO 2: DISSONÂNCIA SEMÂNTICA (O CASO DO CONTRASTE)
- **ENTRADA DE CV**: Objetos: [10 xícaras de café, 1 laptop]. Emoções: [tristeza, exaustão].
- **LEGENDA**: "Dia super produtivo e animado!"
- **SAÍDA**:
    - **DETECÇÃO VISUAL**: Laptop e alta densidade de recipientes de café.
    - **ANÁLISE BIOMÉTRICA**: Expressão facial de exaustão e tristeza.
    - **CONTEXTO TEXTUAL**: Afirmação de alta produtividade e ânimo.
    - **SÍNTESE SEMÂNTICA**: Dissonância crítica. Os indicadores biológicos sugerem fadiga extrema, contradizendo o termo "animado".
    - **OBSERVAÇÕES DE DISSONÂNCIA**: Contradição severa entre fadiga detectada e ânimo relatado. Nota: 9/10.

### EXEMPLO 3: ANÁLISE SEM LEGENDA (CENA PURAMENTE VISUAL)
- **ENTRADA DE CV**: Objetos: [1 laptop, 1 cadeira]. Emoções: [foco, neutralidade].
- **LEGENDA**: "Nenhuma legenda fornecida para esta imagem."
- **SAÍDA**:
    - **DETECÇÃO VISUAL**: Laptop e cadeira, sugerindo ambiente de trabalho ou estudo.
    - **ANÁLISE BIOMÉTRICA**: Expressão neutra indicando concentração.
    - **CONTEXTO TEXTUAL**: Legenda não fornecida.
    - **SÍNTESE SEMÂNTICA**: A análise puramente visual aponta para um cenário de foco produtivo. Há forte correlação entre os equipamentos detectados e a emoção biométrica rastreada.
    - **OBSERVAÇÕES DE DISSONÂNCIA**: Não aplicável (Contexto textual ausente).
"""

# Define o caminho onde o script orquestrador vai buscar
caminho_instrucoes = '/content/instructions.md'

with open(caminho_instrucoes, 'w', encoding='utf-8') as f:
    f.write(conteudo_md)

print(f"Arquivo {caminho_instrucoes} criado com sucesso!")

Arquivo /content/instructions.md criado com sucesso!


In [ ]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_USE_LEGACY_KERAS'] = '1'
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Ollama

In [ ]:
import subprocess
import time

# Tenta encerrar instâncias anteriores para evitar conflitos
!pkill ollama

with open("ollama.log", "w") as log_file:
    process = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

print("⏳ Aguardando o servidor Ollama iniciar...")
time.sleep(10)

!curl -s http://localhost:11434/api/tags > /dev/null && echo "Servidor Ollama online!" || echo "Falha ao iniciar o servidor."

⏳ Aguardando o servidor Ollama iniciar...
Servidor Ollama online!


In [ ]:
!ollama pull gemma2:2b

# Desenvolvimento

1. **Camada de Visão (YOLOv8):** Responsável por identificar objetos físicos (pessoas, carros, acessórios). Se a confiança for baixa, o sistema aplica uma Redução Dinâmica para tentar capturar contexto relevante sem gerar falsos positivos.

2. **Camada Biométrica (DeepFace):** Focada em analisar as faces detectadas utilizando o backend retinaface. O sistema calcula a média emocional do grupo para definir o 'clima' da cena.

3. **Camada de Fusão (LLM):** O script orquestrador une as detecções visuais com a legenda do usuário e submete ao modelo Gemma:2b através do LangChain, buscando identificar concordâncias ou dissonâncias semânticas.

## YOLOv8

In [ ]:
from ultralytics import YOLO

model_yolo_nano = YOLO('yolov8n.pt')

# Usando stream=True para evitar que os resultados se acumulem na RAM e causem crash em vídeos
results = model_yolo_nano.predict(source=caminho_projeto, conf=0.25, stream=True)

# Como results agora é um gerador, precisamos iterar para executar a predição
for r in results:
    pass

print("Varredura inicial concluída com sucesso (Modo Stream).")


image 1/5 /content/drive/MyDrive/notebookcolab/images/people_neutral.jpg: 448x640 1 person, 53.1ms
image 2/5 /content/drive/MyDrive/notebookcolab/images/people_runs.jpg: 448x640 2 persons, 6.7ms
image 3/5 /content/drive/MyDrive/notebookcolab/images/peoples.jpg: 480x640 10 persons, 52.7ms
image 4/5 /content/drive/MyDrive/notebookcolab/images/rage people.jpg: 384x640 1 person, 1 cup, 1 chair, 1 potted plant, 47.1ms
video 5/5 (frame 1/500) /content/drive/MyDrive/notebookcolab/images/peoples_walking.mp4: 448x640 11 persons, 6 chairs, 1 potted plant, 1 dining table, 7.4ms
video 5/5 (frame 2/500) /content/drive/MyDrive/notebookcolab/images/peoples_walking.mp4: 448x640 11 persons, 6 chairs, 1 dining table, 14.3ms
video 5/5 (frame 3/500) /content/drive/MyDrive/notebookcolab/images/peoples_walking.mp4: 448x640 14 persons, 1 backpack, 4 chairs, 1 potted plant, 2 dining tables, 9.0ms
video 5/5 (frame 4/500) /content/drive/MyDrive/notebookcolab/images/peoples_walking.mp4: 448x640 14 persons, 4 ch

### Contagem de objetos

In [ ]:
from collections import Counter

class ObjectDetected:
    def __init__(self, name, confidence):
        self.name = name
        self.confidence = confidence

# Função para extrair os dados brutos do YOLO e converter para sua estrutura
def formatar_objetos_yolo(resultados_yolo):
    # Filtrar por confiança (limite de 50%)
    objetos_filtrados = [
        obj.name for obj in resultados_yolo
        if obj.confidence >= 0.5
    ]

    # Se a lista estiver vazia, aplicar Redução Dinâmica (30%)
    if not objetos_filtrados:
        objetos_filtrados = [
            obj.name for obj in resultados_yolo
            if obj.confidence >= 0.3
        ]
        status = " (Confiança Reduzida)"
    else:
        status = ""

    if not objetos_filtrados:
        return "Nenhum objeto relevante detectado."

    # Contar ocorrências
    contagem = Counter(objetos_filtrados)

    # Montar a string descritiva
    itens = [f"{qtd} {nome}" for nome, qtd in contagem.items()]
    descricao = ", ".join(itens)

    return f"Objetos detectados{status}: {descricao}."

def extrair_dados_yolo(caminho_imagem, modelo):
    # Adicionando parâmetros para evitar NMS Time Limit e gerenciar memória
    # max_det=100 limita o número de caixas para evitar sobrecarga no processamento de sobreposição
    resultados_brutos_gen = modelo.predict(
        source=caminho_imagem,
        conf=0.3,
        verbose=False,
        stream=True,
        max_det=100
    )

    lista_convertida = []
    todos_resultados = []

    for r in resultados_brutos_gen:
        todos_resultados.append(r)
        nomes_classes = r.names

        for box in r.boxes:
            id_classe = int(box.cls[0])
            conf = float(box.conf[0])
            nome = nomes_classes[id_classe]

            lista_convertida.append(ObjectDetected(nome, conf))

    return lista_convertida, todos_resultados

In [ ]:
modelo_nano = YOLO('yolov8n.pt')

# Busca a primeira imagem JPG ou PNG na pasta do projeto para testar com segurança
import glob
imagens_disponiveis = glob.glob(os.path.join(caminho_projeto, '*.jpg')) + glob.glob(os.path.join(caminho_projeto, '*.png'))

if imagens_disponiveis:
  caminho_imagem_teste = imagens_disponiveis[0]
  print(f"Testando inferência YOLOv8 com a imagem: {caminho_imagem_teste}")
  dados_brutos, _ = extrair_dados_yolo(caminho_imagem_teste, modelo_nano)
  string_para_o_prompt = formatar_objetos_yolo(dados_brutos)
  print(string_para_o_prompt)
else:
  print("Nenhuma imagem para teste encontrada na pasta do projeto.")

Testando inferência YOLOv8 com a imagem: /content/drive/MyDrive/notebookcolab/images/people_runs.jpg
Objetos detectados: 2 person.


## DeepFace

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
# Força o TensorFlow (usado pelo DeepFace) a rodar estritamente na CPU,
# liberando 100% da VRAM da GPU T4 para o YOLOv8 e a LLM no Ollama.
tf.config.set_visible_devices([], 'GPU')

from deepface import DeepFace
import cv2

def extracao_de_emocoes(caminho_imagem):
    try:
        # Detecção inicial
        # O DeepFace retorna uma lista de resultados
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False # Evita erro se nenhuma face for encontrada
        )

        if not resultados:
            return "Nenhuma expressão facial detectada."

        face = resultados[0]
        emocao_detectada = face['dominant_emotion']
        confianca = face['emotion'][emocao_detectada] / 100 # O DeepFace fornece a pontuação de todas as emoções; pegamos a dominante

        # Verificação de Limite Fixo (0.5 ou 50%)
        if confianca >= 0.5:
            return f"Emoção: {emocao_detectada} (Alta Confiança: {confianca:.2%})"

        # Redução Dinâmica (Se falhar no passo anterior)
        elif confianca >= 0.3:
            return f"Emoção: {emocao_detectada} (Confiança Reduzida: {confianca:.2%} - Tratar como hipótese)"

        # Falha Crítica
        else:
            return "Nenhuma expressão facial detectada com clareza suficiente."

    except Exception as e:
        return f"Erro no processamento de imagem: {str(e)}"

26-05-25 19:40:38 - Directory /root/.deepface has been created
26-05-25 19:40:38 - Directory /root/.deepface/weights has been created


### Múltiplas *faces*

In [ ]:
from deepface import DeepFace
from collections import Counter

def analisar_clima_do_grupo(caminho_imagem, detector_backend='retinaface'):
    try:
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False,
            detector_backend=detector_backend
        )

        if not isinstance(resultados, list) or len(resultados) == 0:
            return "Nenhuma expressão facial detectada para análise de grupo.", []

        num_faces = len(resultados)
        soma_emocoes = {'angry': 0, 'disgust': 0, 'fear': 0, 'happy': 0, 'sad': 0, 'surprise': 0, 'neutral': 0}

        for face in resultados:
            if 'emotion' in face:
                for emocao, percentual in face['emotion'].items():
                    soma_emocoes[emocao] += percentual

        # Média das emoções
        media_emocoes = {e: (v / num_faces) for e, v in soma_emocoes.items()}

        top_emocoes = sorted(media_emocoes.items(), key=lambda item: item[1], reverse=True)[:2]

        emocao_1, val_1 = top_emocoes[0]
        emocao_2, val_2 = top_emocoes[1]

        desc = f"Expressões Faciais Predominantes: {emocao_1} ({val_1:.1f}%) e {emocao_2} ({val_2:.1f}%) baseado em {num_faces} faces."
        return desc, resultados

    except Exception as e:
        return f"Erro ao processar clima do grupo: {str(e)}", []

## Montagem de prompt e comunicação com LLM

In [ ]:
import re

def montagem_de_prompt(caminho_instrucoes, info_yolo, info_deepface, legenda_usuario):
    # Sanitização básica para evitar injeção de prompt
    legenda_sanitizada = str(legenda_usuario).replace('"', '\\"').replace('\n', ' ')
    legenda_sanitizada = re.sub(r'[<>{}\[\]]', '', legenda_sanitizada)

    # 1. Carregamento e Validação das Instruções
    try:
        with open(caminho_instrucoes, 'r') as f:
            instrucoes = f.read()
            # Validação de integridade das tags que definimos
            if "### INSTRUÇÕES" not in instrucoes:
                raise ValueError("Arquivo de instruções inválido: Tag '### INSTRUÇÕES' não encontrada.")
    except FileNotFoundError:
        return "ERRO: O arquivo instructions.md não foi encontrado no caminho especificado."

    # 2. Montagem Estruturada do Prompt
    # Usamos f-strings para injetar as variáveis nas tags correspondentes
    super_prompt = f"{instrucoes}\n\n"
    super_prompt += "### DADOS DA IMAGEM\n"
    super_prompt += f"{info_yolo}\n"
    super_prompt += f"{info_deepface}\n\n"
    super_prompt += f"### LEGENDA ORIGINAL\n\"{legenda_sanitizada}\"\n\n"
    super_prompt += "### ANÁLISE TÉCNICA\n"

    return super_prompt

In [ ]:
# Comunicação entre meu prompt com a LLM, usando langchain_chain (minha ponte)
def gerar_analise_tecnica(prompt_completo, langchain_chain):
    resposta = langchain_chain.invoke({"question": prompt_completo})
    return resposta

In [ ]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# Configuração do Modelo Local (Ollama)
llm_model = OllamaLLM(model="gemma2:2b")

# Definição do Template
template = """Responda à pergunta baseada nas instruções técnicas fornecidas.

Pergunta: {question}

Resposta:"""

prompt_template = ChatPromptTemplate.from_template(template)

# Criação da Chain
chain = prompt_template | llm_model

print("✅ Variável 'chain' inicializada com o modelo gemma:2b!")

✅ Variável 'chain' inicializada com o modelo gemma:2b!


## Função de limpeza de gpu

Esta célula irá limpar todo lixo deixado pelo python, além de limpar a sesão do Keras e liberar RAM.

In [ ]:
import torch
import gc
import tensorflow as tf

def limpar_memoria_gpu():
  gc.collect()
  tf.keras.backend.clear_session()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Conclusão

1. **Automação de Relatórios:** A capacidade de transformar dados brutos de visão computacional em documentos PDF técnicos, prontos para análise acadêmica.

2. **Confiabilidade Multimodal:** A integração entre YOLOv8 e DeepFace provou ser eficaz para capturar a complexidade de cenas reais, identificando discrepâncias entre o que é visto e o que é relatado (legenda).

3. **Eficiência Operacional:** Com a implementação de rotinas de limpeza de memória e orquestração via LangChain, o sistema mantém estabilidade mesmo no processamento de lotes de imagens, garantindo a viabilidade do TCC em hardware acessível.

## Configurando relatório

In [ ]:
from fpdf import FPDF
import os
from datetime import datetime

def gerar_pdf_consolidado(lista_resultados_processamento, nome_arquivo_final):
    class PDF(FPDF):
        def header(self):
            # Título Principal
            self.set_font('Arial', 'B', 16)
            self.set_text_color(0, 0, 0) # Black
            self.cell(0, 10, 'RELATÓRIO CONSOLIDADO DE ANÁLISE MULTIMODAL', 0, 1, 'C')

            # Subtítulo ou linha de divisão
            self.set_draw_color(0, 0, 0) # Black
            self.set_line_width(1)
            self.line(10, 22, 200, 22)
            self.ln(6)

        def footer(self):
            # Rodapé com número da página e data
            self.set_y(-15)
            self.set_font('Arial', 'I', 8)
            self.set_text_color(128, 128, 128)
            data_atual = datetime.now().strftime('%d/%m/%Y %H:%M')
            self.cell(100, 10, f'Gerado em {data_atual}', 0, 0, 'L')
            self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'R')

    pdf = PDF()
    pdf.set_auto_page_break(auto=True, margin=20)

    if not lista_resultados_processamento:
        pdf.add_page()
        pdf.set_font('Arial', '', 12)
        pdf.cell(0, 10, 'Nenhum resultado de imagem para gerar o relatório.', 0, 1, 'C')
    else:
        for i, resultado in enumerate(lista_resultados_processamento):
            pdf.add_page()
            # Title for each image section
            pdf.set_font('Arial', 'B', 14)
            pdf.set_text_color(0, 0, 0)
            nome_img_display = os.path.basename(resultado['caminho_imagem']) if resultado['caminho_imagem'] else "N/A"
            pdf.cell(0, 10, f'Análise da Imagem: {nome_img_display}', 0, 1, 'L')
            pdf.ln(5)

            # 1. BLOCO DE METADADOS DA ANÁLISE
            pdf.set_fill_color(245, 247, 250) # Keep light gray fill
            pdf.set_draw_color(210, 220, 230) # Keep light gray border
            pdf.set_line_width(0.5)
            # Adjust rect position based on current y
            current_y = pdf.get_y()
            pdf.rect(10, current_y, 190, 40, 'DF') # Increased height slightly

            pdf.set_xy(12, current_y + 2)
            pdf.set_font('Arial', 'B', 10)
            pdf.set_text_color(0, 0, 0) # Black
            pdf.cell(0, 5, 'METADADOS DO PROCESSAMENTO', 0, 1)

            pdf.set_font('Arial', '', 9)
            pdf.set_text_color(60, 60, 60)
            pdf.set_x(12)
            pdf.cell(0, 5, f'Objetos Detectados (YOLOv8): {resultado["info_yolo"]}', 0, 1)
            pdf.set_x(12)
            pdf.cell(0, 5, f'Clima do Grupo (DeepFace): {resultado["info_deepface"]}', 0, 1)
            pdf.set_x(12)
            pdf.cell(0, 5, f'Legenda do Usuário: {resultado["legenda"] if resultado["legenda"] else "Nenhuma legenda fornecida"}', 0, 1)
            pdf.ln(5) # Space after the block

    caminho_pdf = os.path.join(caminho_projeto, f'{nome_arquivo_final}.pdf')
    pdf.output(caminho_pdf)
    print(f'✅ PDF Relatório Consolidado gerado em: {caminho_pdf}')
    return caminho_pdf


## Pipeline CI/CD

In [ ]:
import os
import glob
import cv2
from tqdm import tqdm

def salvar_visualizacao_deteccao(caminho_img, resultados_yolo, resultados_deepface, pasta_saida='/content/outputs'):
    img = cv2.imread(caminho_img)
    if img is None:
        print(f"  -> Erro ao carregar imagem para anotação: {caminho_img}")
        return None

    os.makedirs(pasta_saida, exist_ok=True)

    # 1. Desenhar YOLO Bounding Boxes
    for r in resultados_yolo:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            cls_name = r.names[cls_id]
            if conf >= 0.3:
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                label = f"YOLO: {cls_name} {conf:.2f}"
                cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    # 2. Desenhar DeepFace Bounding Boxes
    if isinstance(resultados_deepface, list):
        for face in resultados_deepface:
            if 'region' in face:
                region = face['region']
                x, y, w, h = region['x'], region['y'], region['w'], region['h']
                cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2)
                if 'dominant_emotion' in face:
                    emocao = face['dominant_emotion']
                    conf_emocao = face['emotion'][emocao]
                    label_emocao = f"Face: {emocao} ({conf_emocao:.1f}%)"
                    cv2.putText(img, label_emocao, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    nome_saida = os.path.basename(caminho_img)
    caminho_saida = os.path.join(pasta_saida, f"debug_{nome_saida}")
    cv2.imwrite(caminho_saida, img)
    return caminho_saida

def executar_pipeline(caminho_img, modelo_yolo, modelo_llm, caminho_regras, legenda=None, detector_backend='retinaface'):
    print(f"[{os.path.basename(caminho_img)}] Iniciando processamento...")

    texto_legenda = legenda if legenda and str(legenda).strip() != "" else "Nenhuma legenda fornecida para esta imagem."

    # Visão Computacional
    dados_yolo, resultados_yolo = extrair_dados_yolo(caminho_img, modelo_yolo)
    info_yolo = formatar_objetos_yolo(dados_yolo)
    info_deepface, resultados_deepface = analisar_clima_do_grupo(caminho_img, detector_backend=detector_backend)

    # Salvar imagem anotada com Bounding Boxes
    try:
        caminho_anotado = salvar_visualizacao_deteccao(caminho_img, resultados_yolo, resultados_deepface, '/content/outputs')
        if caminho_anotado:
            print(f"  -> Imagem com Bounding Boxes salva em: {caminho_anotado}")
    except Exception as e:
        print(f"  -> Erro ao desenhar Bounding Boxes: {e}")

    # Orquestração de Prompt
    prompt = montagem_de_prompt(caminho_regras, info_yolo, info_deepface, texto_legenda)

    # Inferência LLM
    print("  -> Analisando contexto multimodal com LLM...")
    analise = gerar_analise_tecnica(prompt, modelo_llm)

    # Limpeza de memória
    limpar_memoria_gpu()

    print("  -> Finalizado e Memória Limpa!")
    return {
        'caminho_imagem': caminho_img,
        'info_yolo': info_yolo,
        'info_deepface': info_deepface,
        'legenda': texto_legenda,
        'analise_llm': analise
    }

def processar_lote_imagens(pasta_imagens, modelo_yolo, modelo_llm, caminho_regras, nome_pdf_final="Relatorio_Consolidado_Multimodal",
detector_backend='retinaface'):
    extensoes = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    lista_imagens = []

    for ext in extensoes:
        lista_imagens.extend(glob.glob(os.path.join(pasta_imagens, ext)))

    if not lista_imagens:
        print(f"ATENÇÃO: Nenhuma imagem encontrada na pasta: {pasta_imagens}")
        return

    print(f"Encontradas {len(lista_imagens)} imagens para processar.")

    resultados_para_pdf = []
    for caminho_img in tqdm(lista_imagens, desc="Processando Imagens"):
        nome_arquivo = os.path.basename(caminho_img)
        try:
            resultado_imagem = executar_pipeline(
                caminho_img=caminho_img,
                legenda=None,
                modelo_yolo=modelo_yolo,
                modelo_llm=modelo_llm,
                caminho_regras=caminho_regras,
                detector_backend=detector_backend
            )
            resultados_para_pdf.append(resultado_imagem)
        except Exception as e:
            print(f"  -> Erro crítico ao processar o arquivo '{nome_arquivo}': {e}")

    print("\nProcessamento de imagens concluído. Gerando relatório consolidado...")
    gerar_pdf_consolidado(resultados_para_pdf, nome_pdf_final)
    print("Todos os lotes foram processados com sucesso!")

print("INICIANDO VARREDURA DA PASTA...")
processar_lote_imagens(
    pasta_imagens=caminho_projeto,
    modelo_yolo=model_yolo_nano,
    modelo_llm=chain,
    caminho_regras=caminho_instrucoes
)

INICIANDO VARREDURA DA PASTA...
Encontradas 4 imagens para processar.


Processando Imagens:   0%|          | 0/4 [00:00<?, ?it/s]

[people_runs.jpg] Iniciando processamento...
26-05-25 19:40:48 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5

  0%|          | 0.00/119M [00:00<?, ?B/s]
 22%|██▏       | 25.7M/119M [00:00<00:00, 255MB/s]
 51%|█████     | 60.3M/119M [00:00<00:00, 305MB/s]
100%|██████████| 119M/119M [00:00<00:00, 327MB/s] 


26-05-25 19:40:57 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


...

Processando Imagens:  75%|███████▌  | 3/4 [02:18<00:33, 33.40s/it]

  -> Finalizado e Memória Limpa!
[rage people.jpg] Iniciando processamento...
  -> Imagem com Bounding Boxes salva em: /content/outputs/debug_rage people.jpg
  -> Analisando contexto multimodal com LLM...


Processando Imagens: 100%|██████████| 4/4 [02:28<00:00, 37.14s/it]

  -> Finalizado e Memória Limpa!

Processamento de imagens concluído. Gerando relatório consolidado...


✅ PDF Relatório Consolidado gerado em: /content/drive/MyDrive/notebookcolab/images/Relatorio_Consolidado_Multimodal.pdf
Todos os lotes foram processados com sucesso!


### Análise de Vídeo
Esta seção permite processar arquivos de vídeo (.mp4, .avi, etc), extraindo frames periodicamente para análise multimodal.

In [ ]:
import cv2
import os

def processar_video(caminho_video, modelo_yolo, modelo_llm, caminho_regras, intervalo_segundos=2, detector_backend='opencv'):
    """
    Extrai frames de um vídeo e processa cada um através do pipeline com limpeza agressiva de memória.
    """
    if not os.path.exists(caminho_video):
        print(f"Erro: Vídeo não encontrado em {caminho_video}")
        return

    video = cv2.VideoCapture(caminho_video)
    fps = video.get(cv2.CAP_PROP_FPS)
    intervalo_frames = int(fps * intervalo_segundos)

    resultados_video = []
    frame_count = 0
    sucesso = True

    nome_video = os.path.splitext(os.path.basename(caminho_video))[0]
    print(f"Iniciando análise otimizada do vídeo: {nome_video}")

    while sucesso:
        sucesso, frame = video.read()

        if sucesso and frame_count % intervalo_frames == 0:
            caminho_temp_frame = f"/content/temp_frame_{frame_count}.jpg"
            cv2.imwrite(caminho_temp_frame, frame)

            segundo = int(frame_count/fps)
            print(f"\n--- Analisando Segundo {segundo} ---")

            try:
                resultado = executar_pipeline(
                    caminho_img=caminho_temp_frame,
                    modelo_yolo=modelo_yolo,
                    modelo_llm=modelo_llm,
                    caminho_regras=caminho_regras,
                    detector_backend=detector_backend
                )
                resultado['caminho_imagem'] = f"{nome_video} (Segundo {segundo})"
                resultados_video.append(resultado)
            except Exception as e:
                print(f"Erro no frame {frame_count}: {e}")

            # Limpeza imediata pós-frame
            if os.path.exists(caminho_temp_frame):
                os.remove(caminho_temp_frame)

            # Força a limpeza de cache e lixo de memória para evitar o crash
            limpar_memoria_gpu()

        frame_count += 1

    video.release()

    if resultados_video:
        print(f"\nAnálise concluída. Gerando relatório para {len(resultados_video)} frames...")
        gerar_pdf_consolidado(resultados_video, f"Relatorio_Video_{nome_video}")
        # Limpeza final
        limpar_memoria_gpu()
    else:
        print("Nenhum frame foi processado com sucesso.")

### Executar Análise de Vídeo
Use a célula abaixo para processar vídeos encontrados na pasta do projeto.

In [ ]:
import glob

# Busca arquivos de vídeo comuns na pasta do projeto
extensoes_video = ('*.mp4', '*.avi', '*.mov', '*.mkv')
lista_videos = []

for ext in extensoes_video:
    lista_videos.extend(glob.glob(os.path.join(caminho_projeto, ext)))

if not lista_videos:
    print(f"Nenhum vídeo encontrado em: {caminho_projeto}")
else:
    print(f"Encontrado(s) {len(lista_videos)} vídeo(s). Iniciar processamento?")
    for video_path in lista_videos:
        # Processa o vídeo (ajuste o intervalo_segundos se necessário)
        processar_video(
            caminho_video=video_path,
            modelo_yolo=model_yolo_nano,
            modelo_llm=chain,
            caminho_regras=caminho_instrucoes,
            intervalo_segundos=5 # Analisa 1 frame a cada 5 segundos para maior rapidez
        )

Encontrado(s) 1 vídeo(s). Iniciar processamento?
Iniciando análise otimizada do vídeo: peoples_walking

--- Analisando Segundo 0 ---
[temp_frame_0.jpg] Iniciando processamento...
  -> Imagem com Bounding Boxes salva em: /content/outputs/debug_temp_frame_0.jpg
  -> Analisando contexto multimodal com LLM...
  -> Finalizado e Memória Limpa!

--- Analisando Segundo 5 ---
[temp_frame_125.jpg] Iniciando processamento...
  -> Imagem com Bounding Boxes salva em: /content/outputs/debug_temp_frame_125.jpg
  -> Analisando contexto multimodal com LLM...
  -> Finalizado e Memória Limpa!

--- Analisando Segundo 10 ---
[temp_frame_250.jpg] Iniciando processamento...
  -> Imagem com Bounding Boxes salva em: /content/outputs/debug_temp_frame_250.jpg
  -> Analisando contexto multimodal com LLM...
  -> Finalizado e Memória Limpa!

--- Analisando Segundo 15 ---
[temp_frame_375.jpg] Iniciando processamento...
  -> Imagem com Bounding Boxes salva em: /content/outputs/debug_temp_frame_375.jpg
  -> Analisando

## Resultados

Este notebook apresenta uma arquitetura robusta para análise multimodal de imagens e vídeos, integrando Visão Computacional (YOLOv8 e DeepFace) com um Modelo de Linguagem Grande (LLM) orquestrado via LangChain. A solução demonstra eficiência na detecção de objetos, análise de emoções e fusão contextual, culminando na geração de relatórios consolidados em PDF.

### 1. Preparação e Configuração do Ambiente
- **Instalação de Dependências**: As bibliotecas essenciais como `ultralytics` (para YOLOv8), `deepface` (para análise facial), `langchain-ollama` (para interação com LLM local), e `fpdf` (para geração de relatórios) são instaladas, garantindo um ambiente completo para as operações.
- **Ollama**: O servidor Ollama é iniciado localmente e o modelo `gemma2:2b` é carregado, permitindo a inferência da LLM sem dependência de APIs externas ou serviços de nuvem.
- **Google Drive Integration**: O Google Drive é montado para persistência de dados, com uma pasta (`/content/drive/MyDrive/notebookcolab/images`) criada para armazenar as mídias a serem analisadas.
- **Arquivo de Instruções (`instructions.md`)**: Um arquivo de `markdown` é gerado programaticamente, contendo um `prompt` detalhado que define a persona, o contexto operacional e as diretrizes de comportamento para o LLM. Este arquivo é crucial para guiar a análise semântica da LLM, incluindo regras de resolução de conflitos e exemplos de referência (*few-shot*).

### 2. Camadas de Visão Computacional
- **YOLOv8 (Detecção de Objetos)**:
    - O modelo `yolov8n.pt` é carregado para detecção de objetos.
    - A predição utiliza `stream=True` para otimização de memória, crucial ao lidar com vídeos e grandes lotes de imagens.
    - A função `extrair_dados_yolo` é responsável por processar os resultados brutos do YOLO, aplicando filtragem por confiança (padrão de 50%, com redução dinâmica para 30% se a detecção inicial for escassa).
    - A função `formatar_objetos_yolo` agrega e formata os objetos detectados para um formato legível para o LLM.
    - **Resultado Adquirido**: Capacidade de identificar e quantificar objetos em imagens e frames de vídeo, com tratamento adaptativo para diferentes níveis de confiança.

- **DeepFace (Análise Biométrica)**:
    - O `DeepFace` é configurado para rodar exclusivamente na CPU (`tf.config.set_visible_devices([], 'GPU')`), liberando a VRAM da GPU para o YOLOv8 e o Ollama.
    - A função `analisar_clima_do_grupo` processa imagens para detectar múltiplas faces e calcular a média das emoções predominantes (ex: `happy`, `sad`, `neutral`) utilizando o *backend* `retinaface` para detecção facial robusta.
    - **Resultado Adquirido**: Extração precisa de expressões faciais e inferência do 'clima emocional' geral de grupos de pessoas, mesmo em cenários com múltiplas faces.

### 3. Orquestração e Fusão Multimodal (LLM)
- **Montagem do Prompt**: A função `montagem_de_prompt` sanitiza as entradas e constrói um `super_prompt` concatenando as instruções definidas em `instructions.md`, os dados de detecção visual (YOLO), a análise biométrica (DeepFace) e a legenda fornecida pelo usuário.
- **Comunicação com LLM (LangChain)**:
    - A biblioteca `langchain-ollama` é utilizada para interagir com o modelo `gemma2:2b` hospedado localmente pelo Ollama.
    - Uma `ChatPromptTemplate` é definida e uma `chain` é criada para orquestrar a comunicação entre o `prompt` e a `llm_model`.
    - A função `gerar_analise_tecnica` invoca a `chain` com o `prompt` completo, obtendo a análise contextual da LLM.
    - **Resultado Adquirido**: Integração eficaz entre as camadas de visão computacional e o LLM, permitindo uma análise semântica profunda e contextualizada dos elementos visuais e textuais.

### 4. Gerenciamento de Memória
- **Limpeza de GPU (`limpar_memoria_gpu`)**: Uma função dedicada é implementada para realizar uma limpeza agressiva de memória, coletando lixo do Python, limpando a sessão do Keras e esvaziando o cache da CUDA (se disponível). Isso é crucial para manter a estabilidade do ambiente, especialmente durante o processamento de vídeos e lotes de imagens, prevenindo *out-of-memory errors*.

### 5. Geração de Relatórios Consolidado (PDF)
- **`gerar_pdf_consolidado`**: Esta função utiliza `FPDF` para criar relatórios em PDF formatados. Cada página do relatório contém a análise de uma imagem ou frame de vídeo, incluindo:
    - Metadados do processamento (objetos YOLO, emoções DeepFace, legenda).
    - A análise técnica gerada pela LLM.
    - **Resultado Adquirido**: Automatização da criação de relatórios técnicos detalhados, prontos para documentação e análise.

### 6. Pipeline CI/CD (Processamento de Lotes e Vídeos)
- **Visualização com Bounding Boxes (`salvar_visualizacao_deteccao`)**: Esta função desenha as caixas delimitadoras (YOLO) e as caixas faciais (DeepFace) nas imagens originais, salvando as imagens anotadas em `/content/outputs/` para visualização e depuração. Isso fornece um feedback visual imediato das detecções.
- **Processamento de Lotes de Imagens (`processar_lote_imagens`)**: A função itera sobre todas as imagens JPEG e PNG em uma pasta, aplicando o `pipeline` completo (`executar_pipeline`) a cada uma, gerando a análise multimodal e consolidando os resultados em um único PDF.
- **Processamento de Vídeos (`processar_video`)**: Esta função extrai frames de vídeos em intervalos configuráveis (ex: a cada 5 segundos), aplica o `pipeline` completo a cada frame, e garante a limpeza de memória após o processamento de cada frame para otimizar o uso de recursos. Os resultados são também compilados em um PDF específico para o vídeo.
- **Testes Práticos**: O notebook demonstra com sucesso o processamento de um lote de imagens e um vídeo (`peoples_walking.mp4`), gerando os relatórios PDF correspondentes e as imagens com *bounding boxes*.
    - As saídas mostram a detecção de objetos (`person`, `chair`, `dining table`, `potted plant`, etc.) e a análise de emoções, confirmando a funcionalidade de cada camada.
    - O processo de vídeo exemplifica a capacidade de analisar cenas dinâmicas e o gerenciamento eficaz de recursos.

### 7. Futuras Features
- **Captura de Imagem via Webcam**: O notebook inclui o *boilerplate* para uma funcionalidade de captura de imagem via webcam utilizando uma ponte JavaScript (`capturar_foto_webcam`), indicando planos para estender a interatividade do sistema.

### Conclusão dos Resultados
O projeto demonstra uma solução integrada e funcional para análise multimodal, validando a sinergia entre modelos de visão computacional e LLMs. A capacidade de gerar relatórios estruturados e visualizar as detecções, aliada a um gerenciamento cuidadoso de recursos, posiciona este trabalho como uma base sólida para futuras expansões e aplicações em cenários do mundo real.

# Futuras Features

Próximas implementações do TCC, focando em visualização, interatividade e enriquecimento de dados.

### 2. Captura de Imagem via Webcam (JavaScript Bridge)

Como o Colab roda em nuvem, usamos JavaScript para acessar a câmera local do navegador.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def capturar_foto_webcam(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename